In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler


nasa_df = pd.read_csv("Nasa_data.csv")
nasa_df

,event_id,title,category,source,date,latitude,longitude,magnitude_value,magnitude_unit,closed
0,EONET_19349,"Holly Springs IU 81-1 83-1 RX Prescribed Fire,...",Wildfires,IRWIN,2026-04-06T09:15:00Z,34.497590,-89.382263,2113.0,acres,NaN
1,EONET_19351,"EASTER PASTURE Wildfire, Hendry, Florida",Wildfires,IRWIN,2026-04-05T22:32:00Z,26.471389,-81.026944,562.0,acres,NaN
2,EONET_19329,Tropical Cyclone Vaianu,Severe Storms,JTWC,2026-04-05T00:00:00Z,-13.800000,171.800000,40.0,kts,NaN
3,EONET_19329,Tropical Cyclone Vaianu,Severe Storms,JTWC,2026-04-05T06:00:00Z,-14.500000,172.300000,55.0,kts,NaN
4,EONET_19329,Tropical Cyclone Vaianu,Severe Storms,JTWC,2026-04-05T12:00:00Z,-14.900000,172.400000,60.0,kts,NaN
...,...,...,...,...,...,...,...,...,...,...
1053,EONET_14663,"STUD HORSE Wildfire, Okanogan, Washington",Wildfires,IRWIN,2025-07-31T18:44:00Z,48.475167,-120.149667,539.0,acres,NaN
1054,EONET_14656,"Lake Creek Wildfire, Umatilla, Oregon",Wildfires,IRWIN,2025-07-31T17:47:00Z,45.436700,-118.625567,500.0,acres,NaN
1055,EONET_14657,"Lightning Creek Wildfire, Bonner, Idaho",Wildfires,IRWIN,2025-07-31T12:09:00Z,48.284000,-116.147200,700.0,acres,NaN
1056,EONET_14545,"BU 103-156 Rx 0731 Prescribed Fire, Wakulla, F...",Wildfires,IRWIN,2025-07-31T09:19:00Z,30.124639,-84.170333,3550.0,acres,NaN


In [51]:
nasa_df.isnull().sum()

event_id              0
title                 0
category              0
source                0
date                  0
latitude              0
longitude             0
magnitude_value      19
magnitude_unit       19
closed             1058
dtype: int64

In [52]:
nasa_df.columns

Index(['event_id', 'title', 'category', 'source', 'date', 'latitude',
       'longitude', 'magnitude_value', 'magnitude_unit', 'closed'],
      dtype='object')

In [65]:
# Remove empty column & fill missing values with median value & adjust the date
def handle_value(nasa_df):
    nasa_df = nasa_df.drop(columns=["closed"], errors="ignore")
    nasa_df = nasa_df.fillna(nasa_df.median(numeric_only=True)) 
    nasa_df['date'] = pd.to_datetime(nasa_df['date']).dt.date
    return nasa_df

def encode_to_numbers(nasa_df):
    category_map = {'Wildfires': 0,'Severe Storms': 1,'Sea and Lake Ice': 2,'Volcanoes': 3}
    source_map = {'IRWIN': 0,'JTWC': 1,'NATICE': 2,'SIVolcano': 3}

    nasa_df['category_encoded'] = nasa_df['category'].map(category_map)
    nasa_df['source_encoded'] = nasa_df['source'].map(source_map)
    return nasa_df

def extract_location(title):
    if isinstance(title, str):
        return title.split(',')[-1].strip()
    return None

nasa_df = nasa_df.copy()
nasa_df['location'] = nasa_df['title'].apply(extract_location)

nasa_df.head()


,event_id,title,category,source,date,latitude,longitude,magnitude_value,magnitude_unit,closed,location
0,EONET_19349,"Holly Springs IU 81-1 83-1 RX Prescribed Fire,...",Wildfires,IRWIN,2026-04-06T09:15:00Z,34.497590,-89.382263,2113.0,acres,NaN,Mississippi
1,EONET_19351,"EASTER PASTURE Wildfire, Hendry, Florida",Wildfires,IRWIN,2026-04-05T22:32:00Z,26.471389,-81.026944,562.0,acres,NaN,Florida
2,EONET_19329,Tropical Cyclone Vaianu,Severe Storms,JTWC,2026-04-05T00:00:00Z,-13.800000,171.800000,40.0,kts,NaN,Tropical Cyclone Vaianu
3,EONET_19329,Tropical Cyclone Vaianu,Severe Storms,JTWC,2026-04-05T06:00:00Z,-14.500000,172.300000,55.0,kts,NaN,Tropical Cyclone Vaianu
4,EONET_19329,Tropical Cyclone Vaianu,Severe Storms,JTWC,2026-04-05T12:00:00Z,-14.900000,172.400000,60.0,kts,NaN,Tropical Cyclone Vaianu


In [54]:
nasa_df.groupby('magnitude_unit')['magnitude_value'].describe()

,count,mean,std,min,25%,50%,75%,max
magnitude_unit,,,,,,,,
NM^2,14.0,23.007143,9.674841,20.0,20.0,20.0,20.00,56.17
acres,977.0,3022.083930,21344.509408,500.0,700.0,1064.0,1922.00,572804.00
kts,48.0,76.354167,22.091194,35.0,60.0,77.5,91.25,125.00
